<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSEaxoapR7GyB6FC7xp4aM-SrjhxTWTdFCmKAX3TvvFvQ&s=10" alt="Tecnológico de Costa Rica" width="160">

# 🌎 WALC 2026
### Ciencia de Datos · Introducción al Aprendizaje Automático
**Día 4 (Jueves) — No supervisado — Agrupación (K-means) y asociación (Apriori)**

**Instituto Tecnológico de Costa Rica (TEC)** · Instructor: **Efrén Jiménez Delgado**
· 📧 efjimenez@itcr.ac.cr · 🔗 [LinkedIn](https://www.linkedin.com/in/ejimenezdelgado/)

# 🧺 Reglas de asociación (Apriori): la canasta de un supermercado

## 🧵 El hilo conductor (metodología CRISP-DM)

Este cuaderno sigue las seis fases de **CRISP-DM** [1] [2]: cada título de fase marca dónde estamos en el proceso.

| Fase | En este cuaderno |
|---|---|
| 🎯 Comprensión del negocio | Descubrir qué productos de un supermercado se compran juntos para ventas cruzadas y ubicación en góndola. |
| 🔍 Comprensión de los datos | Descargamos 7 501 transacciones de un comercio de alimentos (sin encabezado). |
| 🧹 Preparación de los datos | Convertimos cada fila en una lista de productos y la codificamos como matriz verdadero/falso. |
| 🤖 Modelado | Aplicamos Apriori (soporte mínimo) y generamos las reglas. |
| 📊 Evaluación | Interpretamos soporte, confianza y lift y filtramos las reglas accionables. |
| 🚀 Despliegue y conclusiones | Conclusiones de negocio y límites de las reglas de asociación. |

> 🎯 **Analogía:** el cajero de un supermercado que, de tanto escanear tickets, aprende que "quien lleva espaguetis suele llevar también agua mineral" y propone dónde poner cada cosa.

### 🗺️ Mapa del curso (cronograma Track 5 · Ciencia de datos)
| Día | Jornada | Teoría y laboratorio |
|---|---|---|
| 1 | Lunes | Predicción: regresión lineal y series de tiempo |
| 2 | Martes | Clasificación 1: árboles de decisión, bosques aleatorios y KNN |
| 3 | Miércoles | Clasificación 2: SVM, regresión logística, Naive Bayes y redes neuronales |
| 👉 **4** | Jueves | **No supervisado: agrupación y asociación** |
| 5 | Viernes | Ciencia de datos con IA generativa, ética y buenas prácticas |

# Reglas de asociación (Apriori): análisis de la canasta de un supermercado

El análisis de reglas de asociación es una técnica clave de los minoristas para descubrir qué artículos se compran juntos con frecuencia [4].

### Autores
* Efrén Jiménez

---
## 🎯 Fase 1 · Comprensión del negocio

Descubrir qué productos de un supermercado se compran juntos para ventas cruzadas y ubicación en góndola.

📚 Lectura recomendada: [4]

## Análisis del problema

Las reglas de asociación permiten identificar reglas *fuertes* en datos de transacciones usando medidas de interés [4]. **Pregunta de negocio:** ¿qué productos de un supermercado se compran juntos con tanta frecuencia que conviene ofrecerlos en promoción cruzada o ubicarlos cerca?

### Análisis de la canasta de mercado

El *market basket analysis* estudia qué artículos coexisten en una misma transacción. **Ejemplo:** si un cliente compra pan, es probable que compre mantequilla, mermelada o leche. **Aplicaciones:** venta cruzada, ubicación de productos (complementarios y sustitutos), detección de reacciones adversas a medicamentos en salud y detección de fraude en tarjetas de crédito [9].

### 🧪 Teoría rápida: reglas de asociación

Una **regla de asociación** tiene la forma **A → B** ("quien compra A suele comprar también B") y se descubre en datos de **transacciones** (canastas de compra) [4]. Se evalúa con tres medidas:

| Medida | Fórmula | Se lee como |
|---|---|---|
| **Soporte** de A | $\text{sop}(A)=\dfrac{\#\{\text{transacciones que contienen }A\}}{\#\{\text{transacciones}\}}$ | qué tan frecuente es el conjunto |
| **Confianza** de A → B | $\text{conf}(A\to B)=\dfrac{\text{sop}(A\cup B)}{\text{sop}(A)}$ | probabilidad de B dado A |
| **Elevación** (*lift*) | $\text{lift}(A\to B)=\dfrac{\text{conf}(A\to B)}{\text{sop}(B)}$ | cuántas veces más probable es B si hay A, frente a B sin saber nada |

Un *lift* > 1 indica asociación positiva, = 1 independencia y < 1 que se **sustituyen** [6]. Otras medidas útiles son el **apalancamiento** (*leverage*) $\text{sop}(A\cup B)-\text{sop}(A)\,\text{sop}(B)$ y la **convicción** $\dfrac{1-\text{sop}(B)}{1-\text{conf}(A\to B)}$ [10].

**Algoritmo Apriori.** Se apoya en una propiedad: *todo subconjunto de un conjunto frecuente también es frecuente* (y, al revés, si un conjunto no es frecuente, ninguno de sus superconjuntos lo es), lo que permite **podar** candidatos [5]. El proceso tiene dos pasos: (1) hallar los **conjuntos frecuentes** con soporte ≥ `min_support`; (2) generar las **reglas** con confianza (o lift) ≥ umbral [9]. Alternativas más rápidas son **FP-Growth** [7] y **Eclat** [8].

In [ ]:
%pip install -q squarify mlxtend

In [ ]:
# Manipulación de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import squarify
sns.set_style('whitegrid')

# Reglas de asociación (mlxtend)
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

---
## 🔍 Fase 2 · Comprensión de los datos

Descargamos 7 501 transacciones de un comercio de alimentos (sin encabezado).

📚 Lectura recomendada: [3]

In [ ]:
import os, io, zipfile, urllib.request

# Descarga del conjunto de datos (zip dentro de zip) desde el repositorio del curso en GitLab
if not os.path.exists('Market_Basket_Optimisation.csv'):
    url = 'https://gitlab.com/ejimenez/marketbasketdataset/-/archive/main/marketbasketdataset-main.zip'
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read()))
    interno = [n for n in z.namelist() if n.endswith('Market_Basket_Optimisation.zip')][0]
    zipfile.ZipFile(io.BytesIO(z.read(interno))).extractall('.')
    for raiz, _, archivos in os.walk('.'):          # por si el CSV queda dentro de una carpeta
        if 'Market_Basket_Optimisation.csv' in archivos and raiz != '.':
            os.replace(os.path.join(raiz, 'Market_Basket_Optimisation.csv'), 'Market_Basket_Optimisation.csv')
print('Archivo disponible:', os.path.exists('Market_Basket_Optimisation.csv'))

In [ ]:
# El archivo NO tiene encabezado: cada fila es una transacción
data = pd.read_csv('Market_Basket_Optimisation.csv', header=None)
print('Transacciones y columnas:', data.shape)
data.head()

---
## 🧹 Fase 3 · Preparación de los datos

Convertimos cada fila en una lista de productos y la codificamos como matriz verdadero/falso.

📚 Lectura recomendada: [11]

### Preparación de los datos

Cada fila es una transacción y las celdas vacías (`NaN`) solo rellenan. El `TransactionEncoder` de `mlxtend` [11] necesita **listas de productos** y entrega una matriz verdadero/falso con una columna por producto. Primero se arma cada lista descartando las celdas vacías (en la versión original los vacíos se convertían en el texto `'nan'` y había que borrar esa columna después).

In [ ]:
transacciones = [[p for p in fila if isinstance(p, str)] for fila in data.values.tolist()]
print('Transacciones:', len(transacciones))
print('Productos por transacción: promedio =', round(sum(map(len, transacciones)) / len(transacciones), 2), '| máximo =', max(map(len, transacciones)))

In [ ]:
t = TransactionEncoder()
data = pd.DataFrame(t.fit_transform(transacciones), columns=t.columns_)
print('Matriz transacciones × productos:', data.shape)

In [ ]:
print('¿Hay una columna "nan"?', 'nan' in data.columns)

In [ ]:
r = data.sum(axis=0).sort_values(ascending=False)[:20]
print(f'Producto más comprado: {r.index[0]} ({r.iloc[0] / len(data):.1%} de las transacciones)')
plt.figure(figsize=(12, 5))
sns.barplot(x=r.index, y=r.values, color='steelblue')
plt.xticks(rotation=90)
plt.ylabel('Número de transacciones')
plt.title('20 productos más comprados')
plt.tight_layout()
plt.show()

In [ ]:
cmap = matplotlib.cm.Blues
norm = matplotlib.colors.Normalize(vmin=r.values.min(), vmax=r.values.max())
colors = [cmap(norm(v)) for v in r.values]
plt.figure(figsize=(10, 7))
squarify.plot(sizes=r.values, label=r.index, alpha=.7, color=colors, text_kwargs={'fontsize': 8})
plt.title('Mapa de árbol de los 20 productos más comprados')
plt.axis('off')
plt.show()

---
## 🤖 Fase 4 · Modelado

Aplicamos Apriori (soporte mínimo) y generamos las reglas.

📚 Lectura recomendada: [5]

### Algoritmo Apriori

Buscamos los conjuntos de productos presentes en **al menos el 1 %** de las transacciones (`min_support = 0.01`). Un soporte muy bajo (0.5 %) genera miles de conjuntos poco útiles; uno muy alto deja fuera pares interesantes pero menos frecuentes.

In [ ]:
freq_items = apriori(data, min_support=0.01, use_colnames=True)
freq_items['longitud'] = freq_items['itemsets'].apply(len)
print('Conjuntos frecuentes:', len(freq_items))
print(freq_items['longitud'].value_counts().sort_index().rename('conjuntos').to_frame())

In [ ]:
freq_items.sort_values('support', ascending=False).head(10)

In [ ]:
# Reglas con confianza >= 20 % y asociación positiva (lift > 1)
res = association_rules(freq_items, metric='confidence', min_threshold=0.2)
res = res[res['lift'] > 1].sort_values('lift', ascending=False)
print('Reglas generadas:', len(res))

In [ ]:
res[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)

---
## 📊 Fase 5 · Evaluación

Interpretamos soporte, confianza y lift y filtramos las reglas accionables.

📚 Lectura recomendada: [6] [10]

# Lectura automática de las 3 reglas con mayor lift
for _, f in res.head(3).iterrows():
    print(f"• El {f.confidence:.0%} de las transacciones con {' + '.join(sorted(f.antecedents))} también contienen "
          f"{' + '.join(sorted(f.consequents))} (soporte {f.support:.1%}, lift {f.lift:.2f}).")

In [ ]:
frequent_itemsets = apriori(data, min_support=0.05, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
frequent_itemsets.sort_values('support', ascending=False)

In [ ]:
# Pares de productos (length = 2) con soporte >= 1 %
frequent_itemsets_2 = apriori(data, min_support=0.01, use_colnames=True)
frequent_itemsets_2[frequent_itemsets_2['itemsets'].apply(len) == 2].sort_values('support', ascending=False).head(10)

In [ ]:
# Productos individuales con soporte >= 5 %
frequent_itemsets[frequent_itemsets['length'] == 1]

---
## 🧪 Refuerzo: soporte, confianza y lift calculados a mano

Antes de confiar en la librería, conviene entender de dónde salen los números. Con una canasta mínima de 5 compras calculamos las medidas con conteos y comprobamos que `mlxtend` [11] da lo mismo.

In [ ]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

mini = [['pan', 'leche'], ['pan', 'leche', 'huevos'], ['pan', 'huevos'], ['leche', 'huevos'], ['pan', 'leche', 'cafe']]
n = len(mini)
sop = lambda s: sum(set(s) <= set(t) for t in mini) / n

print('sop(pan)         =', sop(['pan']))
print('sop(pan, leche)  =', sop(['pan', 'leche']))
print('conf(pan→leche)  =', sop(['pan', 'leche']) / sop(['pan']))
print('lift(pan→leche)  =', sop(['pan', 'leche']) / (sop(['pan']) * sop(['leche'])))

te = TransactionEncoder()
M = pd.DataFrame(te.fit_transform(mini), columns=te.columns_)
reglas = association_rules(apriori(M, min_support=0.4, use_colnames=True), metric='confidence', min_threshold=0.5)
reglas[(reglas.antecedents == frozenset({'pan'})) & (reglas.consequents == frozenset({'leche'}))][['support', 'confidence', 'lift']]

### ✍️ Ejercicios
1. Calcule a mano la confianza y el lift de `huevos → leche` y verifique con `mlxtend`.
2. ¿Puede una regla tener confianza alta y lift menor que 1? Construya un ejemplo.
3. Si baja `min_support`, ¿qué pasa con la cantidad de conjuntos frecuentes y con el tiempo de cómputo? [5]

---
## 🧪 Refuerzo: de las reglas a la decisión: filtrar por lift y confianza, y no confundir asociación con causalidad

Una lista con cientos de reglas no sirve para decidir. Un filtro razonable combina: **soporte** suficiente (que la regla afecte a muchos clientes), **confianza** alta (que se cumpla a menudo) y **lift > 1** (que no sea solo porque ambos productos son populares) [10]. Además, las reglas ya generadas pueden **resumirse por consecuente**: ¿qué productos suelen aparecer como "resultado" de otras compras? Esos son candidatos para promociones cruzadas.

In [ ]:
fuertes = res[(res['confidence'] >= 0.3) & (res['lift'] >= 1.2)].copy()
fuertes['regla'] = fuertes['antecedents'].apply(lambda s: ' + '.join(sorted(s))) + '  →  ' + fuertes['consequents'].apply(lambda s: ' + '.join(sorted(s)))
print('Reglas que cumplen confianza ≥ 30 % y lift ≥ 1.2:', len(fuertes))
print(fuertes.sort_values('lift', ascending=False)[['regla', 'support', 'confidence', 'lift']].head(10).round(3).to_string(index=False))

# ¿Qué productos aparecen más veces como consecuente de una regla fuerte?
print(fuertes['consequents'].apply(lambda s: ' + '.join(sorted(s))).value_counts().head(5))

### ✍️ Ejercicios
1. Cambie `min_support` a 0.02 y a 0.003. ¿Cuántas reglas obtiene en cada caso? ¿Qué se gana y qué se pierde?
2. Un producto muy popular (por ejemplo, el agua) aparece en muchas reglas con lift cercano a 1. ¿Por qué el *lift* corrige ese sesgo y la confianza no? [6]
3. Proponga una acción concreta (promoción, ubicación en góndola) para la regla con mayor lift y diga qué experimento (prueba A/B) la validaría.

---
## ✅ Conclusiones y siguientes pasos

Apriori resume miles de tickets en unas pocas reglas legibles, pero hay que usarlas con criterio: el **soporte mínimo** controla cuántas reglas aparecen, el **lift** distingue asociaciones reales de la simple popularidad de un producto [6] y toda regla describe *co-ocurrencia*, no causalidad [10]. Antes de cambiar una góndola o lanzar una promoción conviene validar la regla con un experimento.

---
## 📚 Referencias

Fuentes adicionales para profundizar: [12], [13], [14].

**[1]** Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc. https://www.the-modeling-agency.com/crisp-dm.pdf

**[2]** Wirth, R., & Hipp, J. (2000). CRISP-DM: Towards a standard process model for data mining. *Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining*, 29–39.

**[3]** Jiménez Delgado, E. (s. f.). *marketbasketdataset* [Transacciones de una tienda, 7 501 canastas]. GitLab. https://gitlab.com/ejimenez/marketbasketdataset

**[4]** Agrawal, R., Imieliński, T., & Swami, A. (1993). Mining association rules between sets of items in large databases. *Proceedings of the 1993 ACM SIGMOD International Conference on Management of Data*, 207–216. https://doi.org/10.1145/170035.170072

**[5]** Agrawal, R., & Srikant, R. (1994). Fast algorithms for mining association rules. *Proceedings of the 20th International Conference on Very Large Data Bases (VLDB)*, 487–499.

**[6]** Brin, S., Motwani, R., Ullman, J. D., & Tsur, S. (1997). Dynamic itemset counting and implication rules for market basket data. *Proceedings of the 1997 ACM SIGMOD International Conference on Management of Data*, 255–264.

**[7]** Han, J., Pei, J., & Yin, Y. (2000). Mining frequent patterns without candidate generation. *Proceedings of the 2000 ACM SIGMOD International Conference on Management of Data*, 1–12.

**[8]** Zaki, M. J. (2000). Scalable algorithms for association mining. *IEEE Transactions on Knowledge and Data Engineering, 12*(3), 372–390. https://doi.org/10.1109/69.846291

**[9]** Han, J., Kamber, M., & Pei, J. (2012). *Data Mining: Concepts and Techniques* (3.ª ed.). Morgan Kaufmann. https://doi.org/10.1016/C2009-0-61819-5

**[10]** Tan, P.-N., Steinbach, M., Karpatne, A., & Kumar, V. (2019). *Introduction to Data Mining* (2.ª ed.). Pearson.

**[11]** Raschka, S. (2018). MLxtend: Providing machine learning and data science utilities and extensions to Python's scientific computing stack. *Journal of Open Source Software, 3*(24), 638. https://doi.org/10.21105/joss.00638

**[12]** McKinney, W. (2010). Data structures for statistical computing in Python. *Proceedings of the 9th Python in Science Conference*, 56–61. https://doi.org/10.25080/Majora-92bf1922-00a

**[13]** Waskom, M. L. (2021). seaborn: statistical data visualization. *Journal of Open Source Software, 6*(60), 3021. https://doi.org/10.21105/joss.03021

**[14]** Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering, 9*(3), 90–95. https://doi.org/10.1109/MCSE.2007.55